# 1. Embeddings in PyTorch 

For this we gonna use the 

```python

embedding = nn.Embedding()

```

---

## 1.1 Args For Embedding



```python

nn.Embedding(
    num_embeddings = X,      # (int) Vocab Size
    embedding_dim,       # (int) Dimension of each embedding
    padding_idx=None,    # (int) Index to ignore (complete batchs with differents sizes of tokens)
    max_norm=None,       # (float) Normalize per max_norm, limits the max value 
    norm_type=2.0,       # (float) Type of norm (L2=2.0, L1=1.0)
)

```

---

## 1.2 How It Works Internally

When you create an embedding:

```python
embed = nn.Embedding(vocab_size=50, embedding_dim=768)
```

PyTorch creates a **random matrix (50, 768)**:

```
Row 0  (ID 0)  → [-0.2,  0.5,  0.1, ..., 0.4]  (768 dimensions)
Row 1  (ID 1)  → [ 0.8, -0.3,  0.6, ..., 0.2]
Row 2  (ID 2)  → [ 0.1,  0.2, -0.4, ..., -0.5]
Row 3  (ID 3)  → [-0.5,  0.9,  0.2, ..., 0.7]
...
Row 29 (ID 29) → [ 0.1, 0.2, -0.4, ..., 0.3]
...
Row 49 (ID 49) → [-0.1, 0.3,  0.5, ..., -0.2]
```

**Position in the matrix = Token ID**


### Practical Example


```python
import torch
import torch.nn as nn

# Create embedding: 5 words → 3 dimensions
embed = nn.Embedding(num_embeddings=5, embedding_dim=3)

print(embed.weight)
# Internal matrix (5, 3):
# tensor([[-0.2,  0.5,  0.1],
#         [ 0.8, -0.3,  0.6],
#         [ 0.1,  0.2, -0.4],
#         [-0.5,  0.9,  0.2],
#         [ 0.3, -0.1,  0.7]])

# Pass token IDs
tokens = torch.tensor([1, 3, 0])

# Lookup in table
output = embed(tokens)

# Input (3, )
# Result (3, 3):
# tensor([[ 0.8, -0.3,  0.6],   ← row 1
#         [-0.5,  0.9,  0.2],   ← row 3
#         [-0.2,  0.5,  0.1]])  ← row 0
```


### Manual Equivalent

```python
# Create embedding manually:
embedding_matrix = torch.randn(5, 3)
tokens = torch.tensor([1, 3, 0])

# Do lookup:
output = embedding_matrix[tokens]  # Simple indexing!

# Same thing nn.Embedding does!
```
---

# 2. Types


## 2.1 Absolute Positional Embedding

Embedding + Positional Embedding





## 2.2 RoPE

Rotation the dimensions of the tokens in pairs



In [ ]:
import torch

def build_rope_cache(seq_len, head_dim, base=10000.0, device=None, dtype=torch.float32):

    # Calculate the speed of rotation of each embedding pair, it will be a list
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))

    # Get the INDEX of each token
    t = torch.arange(seq_len, device=device).float()

    # Calculate the embeddings pair wit their rotation
    freqs = torch.outer(t, inv_freq)              

    # 
    emb = torch.cat((freqs, freqs), dim=-1)       # (seq, head_dim)
    return emb.cos().to(dtype), emb.sin().to(dtype)


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)                   # metade i e metade i + d/2
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin, position_ids=None):
    # x: (batch, heads, seq, head_dim)
    if position_ids is not None:                  # (batch, seq), usado com KV cache
        cos = cos[position_ids].unsqueeze(1)      # (batch, 1, seq, head_dim)
        sin = sin[position_ids].unsqueeze(1)
    else:
        cos = cos[None, None, : x.size(2)]        # (1, 1, seq, head_dim)
        sin = sin[None, None, : x.size(2)]
    return x * cos + rotate_half(x) * sin